# AthleteLoad: 02. Interpretable Baseline Modeling

This notebook establishes an interpretable linear baseline using Logistic Regression with balanced class weights on a strictly chronological train/validation/test split.

In [ ]:
import os
import sys
# Ensure parent directory is in python path
if os.path.abspath('..') not in sys.path:
    sys.path.insert(0, os.path.abspath('..'))

def get_path(rel_path):
    if os.path.exists(rel_path):
        return rel_path
    elif os.path.exists(os.path.join('..', rel_path)):
        return os.path.join('..', rel_path)
    return rel_path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

from src.utils import setup_plotting_style, PALETTE
from src.train import run_full_training_pipeline
from src.evaluate import evaluate_binary_model, find_optimal_threshold
setup_plotting_style()

artifacts = run_full_training_pipeline(data_path=get_path('data/processed/player_daily_features.csv'), seed=42)
lr_model = artifacts['models']['Logistic Regression']
feature_cols = artifacts['feature_cols']
X_val_scaled = artifacts['X_val_scaled']
y_val = artifacts['y_val']
X_test_scaled = artifacts['X_test_scaled']
y_test = artifacts['y_test']
print('Logistic Regression Model Loaded.')

## 1. Standardized Coefficients

In [ ]:
coefs = lr_model.coef_[0]
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': coefs,
    'abs_coef': np.abs(coefs)
}).sort_values('abs_coef', ascending=False)

print('Top 10 Influential Features in Logistic Baseline:')
display(coef_df.head(10))

plt.figure(figsize=(10, 6))
top_coefs = pd.concat([coef_df[coef_df['coefficient'] > 0].head(7), coef_df[coef_df['coefficient'] < 0].head(7)]).sort_values('coefficient')
colors = [PALETTE['healthy'] if c < 0 else PALETTE['danger'] for c in top_coefs['coefficient']]
sns.barplot(x='coefficient', y='feature', data=top_coefs, palette=colors, hue='feature', legend=False)
plt.title('Logistic Regression Standardized Coefficients (Top Risk vs Protective Factors)', fontsize=12, fontweight='bold')
plt.xlabel('Standardized Coefficient (Log Odds)', fontweight='bold')
plt.ylabel('Feature', fontweight='bold')
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 2. Evaluation on Test Partition

In [ ]:
val_probs = lr_model.predict_proba(X_val_scaled)[:, 1]
best_thresh, _ = find_optimal_threshold(y_val, val_probs)
print(f'Optimal Validation Threshold: {best_thresh:.3f}')

val_res = evaluate_binary_model(lr_model, X_val_scaled, y_val, threshold=best_thresh)
test_res = evaluate_binary_model(lr_model, X_test_scaled, y_test, threshold=best_thresh)

print(f'Validation: PR-AUC = {val_res["pr_auc"]:.4f} | ROC-AUC = {val_res["roc_auc"]:.4f} | Recall = {val_res["recall"]:.3f} | Prec = {val_res["precision"]:.3f} | F1 = {val_res["f1"]:.3f}')
print(f'Test:       PR-AUC = {test_res["pr_auc"]:.4f} | ROC-AUC = {test_res["roc_auc"]:.4f} | Recall = {test_res["recall"]:.3f} | Prec = {test_res["precision"]:.3f} | F1 = {test_res["f1"]:.3f}')

cm = np.array(test_res['confusion_matrix'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Pred Healthy', 'Pred Injury (7d)'], yticklabels=['Actual Healthy', 'Actual Injury (7d)'])
plt.title('Test Set Confusion Matrix (Logistic Regression)', fontweight='bold')
plt.show()